# PubMedQA Medical Question Answering - Final Colab Notebook

This notebook is **Colab-ready** and reads the dataset from a local **`data/` folder**.

## Expected files inside `data/`
- `ori_pqal.json`
- `test_ground_truth.json`

## Models included
1. TF-IDF + Logistic Regression  
2. TF-IDF + N-grams + Logistic Regression  
3. Sentence-BERT embeddings + Logistic Regression  
4. BioBERT fine-tuning for sequence classification  

## Notes
- `TF-IDF` and `Sentence-BERT` are usually faster.
- `BioBERT` needs a **GPU** in Colab for practical training time.
- This notebook uses:
  - **train/validation** from the part of `ori_pqal.json` **not listed** in `test_ground_truth.json`
  - **test set** from IDs present in `test_ground_truth.json`


In [ ]:
# If running in Colab, install required packages
# You can run this once, then Runtime -> Restart session if needed.

!pip -q install transformers datasets sentence-transformers scikit-learn seaborn accelerate


In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer

from sentence_transformers import SentenceTransformer

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)


In [ ]:
# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Data loading

The notebook first tries these locations:

1. `/content/data`
2. `./data`
3. `/mnt/data` (useful if you test locally in this environment)

In Colab, upload your files so they end up in `/content/data/`.


In [ ]:
# -----------------------------
# Find data folder
# -----------------------------
candidate_dirs = [
    Path("/content/data"),
    Path("./data"),
    Path("/mnt/data"),
]

data_dir = None
for d in candidate_dirs:
    if (d / "ori_pqal.json").exists() and (d / "test_ground_truth.json").exists():
        data_dir = d
        break

if data_dir is None:
    raise FileNotFoundError(
        "Could not find 'ori_pqal.json' and 'test_ground_truth.json'. "
        "Please place both files inside a folder named 'data'."
    )

print("Using data directory:", data_dir.resolve())


In [ ]:
# -----------------------------
# Load JSON files
# -----------------------------
with open(data_dir / "ori_pqal.json", "r", encoding="utf-8") as f:
    full_data = json.load(f)

with open(data_dir / "test_ground_truth.json", "r", encoding="utf-8") as f:
    test_ground_truth = json.load(f)

print("Total examples in ori_pqal.json:", len(full_data))
print("Total test IDs in test_ground_truth.json:", len(test_ground_truth))
print("Overlap:", len(set(full_data.keys()) & set(test_ground_truth.keys())))


In [ ]:
# -----------------------------
# Build dataframe
# -----------------------------
records = []
for pmid, item in full_data.items():
    question = item.get("QUESTION", "")
    contexts = item.get("CONTEXTS", [])
    context = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    label = item.get("final_decision", None)

    records.append({
        "pmid": str(pmid),
        "question": question,
        "context": context,
        "text": f"Question: {question} Context: {context}",
        "label_text": label,
        "year": item.get("YEAR", None),
        "reasoning_required_pred": item.get("reasoning_required_pred", None),
        "reasoning_free_pred": item.get("reasoning_free_pred", None),
    })

df = pd.DataFrame(records)

# Keep only yes/no/maybe
df = df[df["label_text"].isin(["yes", "no", "maybe"])].copy()

label2id = {"yes": 0, "no": 1, "maybe": 2}
id2label = {0: "yes", 1: "no", 2: "maybe"}
df["label"] = df["label_text"].map(label2id)

print(df.shape)
df.head()


In [ ]:
# -----------------------------
# Create train/val/test split using provided test IDs
# -----------------------------
test_ids = set(map(str, test_ground_truth.keys()))

test_df = df[df["pmid"].isin(test_ids)].copy()
trainval_df = df[~df["pmid"].isin(test_ids)].copy()

# Safety: make sure test labels match provided ground truth
test_df["label_text_from_gt"] = test_df["pmid"].map(test_ground_truth)
mismatch_count = (test_df["label_text"] != test_df["label_text_from_gt"]).sum()

print("Train+Val size:", len(trainval_df))
print("Test size:", len(test_df))
print("Label mismatches between ori_pqal and test_ground_truth:", mismatch_count)

# Use GT file as final source of truth for test labels
test_df["label_text"] = test_df["label_text_from_gt"]
test_df["label"] = test_df["label_text"].map(label2id)
test_df.drop(columns=["label_text_from_gt"], inplace=True)

train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.2,
    random_state=SEED,
    stratify=trainval_df["label"]
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

for name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name} label distribution:")
    print(split_df["label_text"].value_counts(normalize=True).round(3))


In [ ]:
# -----------------------------
# Helper functions
# -----------------------------
def evaluate_predictions(y_true, y_pred, split_name="Evaluation", label_order=("yes", "no", "maybe")):
    y_true_labels = [id2label[int(x)] if isinstance(x, (int, np.integer)) else x for x in y_true]
    y_pred_labels = [id2label[int(x)] if isinstance(x, (int, np.integer)) else x for x in y_pred]

    metrics = {
        "accuracy": accuracy_score(y_true_labels, y_pred_labels),
        "macro_f1": f1_score(y_true_labels, y_pred_labels, average="macro"),
        "weighted_f1": f1_score(y_true_labels, y_pred_labels, average="weighted"),
    }

    print(f"\n--- {split_name} ---")
    print(f"Accuracy:    {metrics['accuracy']:.4f}")
    print(f"Macro F1:    {metrics['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true_labels, y_pred_labels, digits=4))

    cm = confusion_matrix(y_true_labels, y_pred_labels, labels=list(label_order))
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=list(label_order),
        yticklabels=list(label_order)
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{split_name} Confusion Matrix")
    plt.show()

    return metrics


def save_prediction_file(df_split, preds, model_name, filename):
    out = df_split.copy().reset_index(drop=True)
    out["predicted_label"] = preds
    out["true_label"] = out["label_text"].values
    out["correct"] = out["predicted_label"] == out["true_label"]
    out.to_csv(filename, index=False)
    print(f"Saved: {filename}")
    return out


## Model 1: TF-IDF + Logistic Regression

In [ ]:
# -----------------------------
# TF-IDF baseline
# -----------------------------
X_train = train_df["text"].tolist()
X_val = val_df["text"].tolist()
X_test = test_df["text"].tolist()

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_clf = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED
)
tfidf_clf.fit(X_train_tfidf, y_train)

val_pred_tfidf = tfidf_clf.predict(X_val_tfidf)
test_pred_tfidf = tfidf_clf.predict(X_test_tfidf)

val_pred_tfidf_labels = [id2label[i] for i in val_pred_tfidf]
test_pred_tfidf_labels = [id2label[i] for i in test_pred_tfidf]

tfidf_val_metrics = evaluate_predictions(y_val, val_pred_tfidf, split_name="TF-IDF Validation")
tfidf_test_metrics = evaluate_predictions(y_test, test_pred_tfidf, split_name="TF-IDF Test")

tfidf_test_out = save_prediction_file(test_df, test_pred_tfidf_labels, "tfidf", "tfidf_test_predictions.csv")


## Model 2: TF-IDF + N-grams + Logistic Regression

In [ ]:
# -----------------------------
# TF-IDF + N-grams baseline
# -----------------------------
tfidf_ngram_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_ngram = tfidf_ngram_vectorizer.fit_transform(X_train)
X_val_ngram = tfidf_ngram_vectorizer.transform(X_val)
X_test_ngram = tfidf_ngram_vectorizer.transform(X_test)

tfidf_ngram_clf = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED
)
tfidf_ngram_clf.fit(X_train_ngram, y_train)

val_pred_ngram = tfidf_ngram_clf.predict(X_val_ngram)
test_pred_ngram = tfidf_ngram_clf.predict(X_test_ngram)

val_pred_ngram_labels = [id2label[i] for i in val_pred_ngram]
test_pred_ngram_labels = [id2label[i] for i in test_pred_ngram]

tfidf_ngram_val_metrics = evaluate_predictions(y_val, val_pred_ngram, split_name="TF-IDF + N-grams Validation")
tfidf_ngram_test_metrics = evaluate_predictions(y_test, test_pred_ngram, split_name="TF-IDF + N-grams Test")

tfidf_ngram_test_out = save_prediction_file(test_df, test_pred_ngram_labels, "tfidf_ngram", "tfidf_ngram_test_predictions.csv")


## Model 3: Sentence-BERT + Logistic Regression

Recommended biomedical Sentence-BERT model:
- `pritamdeka/S-BioBert-snli-multinli-stsb`

If download is slow or unavailable, you can switch to:
- `sentence-transformers/all-MiniLM-L6-v2`


In [ ]:
# -----------------------------
# Sentence-BERT embeddings + classifier
# -----------------------------
SBERT_MODEL_NAME = "pritamdeka/S-BioBert-snli-multinli-stsb"
# SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

X_train_emb = sbert_model.encode(
    X_train,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_val_emb = sbert_model.encode(
    X_val,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_test_emb = sbert_model.encode(
    X_test,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Train embedding shape:", X_train_emb.shape)
print("Val embedding shape:", X_val_emb.shape)
print("Test embedding shape:", X_test_emb.shape)

sbert_clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=SEED
)
sbert_clf.fit(X_train_emb, y_train)

val_pred_sbert = sbert_clf.predict(X_val_emb)
test_pred_sbert = sbert_clf.predict(X_test_emb)

val_pred_sbert_labels = [id2label[i] for i in val_pred_sbert]
test_pred_sbert_labels = [id2label[i] for i in test_pred_sbert]

sbert_val_metrics = evaluate_predictions(y_val, val_pred_sbert, split_name="Sentence-BERT Validation")
sbert_test_metrics = evaluate_predictions(y_test, test_pred_sbert, split_name="Sentence-BERT Test")

np.save("X_train_sbert.npy", X_train_emb)
np.save("X_val_sbert.npy", X_val_emb)
np.save("X_test_sbert.npy", X_test_emb)

sbert_test_out = save_prediction_file(test_df, test_pred_sbert_labels, "sbert", "sbert_test_predictions.csv")


## Model 4: BioBERT fine-tuning

This is the heaviest model in the notebook.  
For Colab:
- use **GPU**
- keep `MAX_LENGTH=512`
- reduce `NUM_EPOCHS` to 2 if needed


In [ ]:
# -----------------------------
# Prepare Hugging Face datasets for BioBERT
# -----------------------------
train_hf = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_hf = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
test_hf = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

BIOBERT_MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
MAX_LENGTH = 512

biobert_tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)

def tokenize_function(batch):
    return biobert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

train_hf_tok = train_hf.map(tokenize_function, batched=True)
val_hf_tok = val_hf.map(tokenize_function, batched=True)
test_hf_tok = test_hf.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=biobert_tokenizer)


In [ ]:
# -----------------------------
# BioBERT model + metrics
# -----------------------------
biobert_model = AutoModelForSequenceClassification.from_pretrained(
    BIOBERT_MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


In [ ]:
# -----------------------------
# BioBERT training
# -----------------------------
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5

training_args = TrainingArguments(
    output_dir="./biobert_pubmedqa_outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=biobert_model,
    args=training_args,
    train_dataset=train_hf_tok,
    eval_dataset=val_hf_tok,
    tokenizer=biobert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
# -----------------------------
# BioBERT validation + test predictions
# -----------------------------
val_output_biobert = trainer.predict(val_hf_tok)
test_output_biobert = trainer.predict(test_hf_tok)

val_pred_biobert = np.argmax(val_output_biobert.predictions, axis=1)
test_pred_biobert = np.argmax(test_output_biobert.predictions, axis=1)

val_pred_biobert_labels = [id2label[i] for i in val_pred_biobert]
test_pred_biobert_labels = [id2label[i] for i in test_pred_biobert]

biobert_val_metrics = evaluate_predictions(y_val, val_pred_biobert, split_name="BioBERT Validation")
biobert_test_metrics = evaluate_predictions(y_test, test_pred_biobert, split_name="BioBERT Test")

biobert_test_out = save_prediction_file(test_df, test_pred_biobert_labels, "biobert", "biobert_test_predictions.csv")


## Final comparison table

In [ ]:
results_summary = pd.DataFrame([
    {
        "model": "TF-IDF",
        "val_accuracy": tfidf_val_metrics["accuracy"],
        "val_macro_f1": tfidf_val_metrics["macro_f1"],
        "val_weighted_f1": tfidf_val_metrics["weighted_f1"],
        "test_accuracy": tfidf_test_metrics["accuracy"],
        "test_macro_f1": tfidf_test_metrics["macro_f1"],
        "test_weighted_f1": tfidf_test_metrics["weighted_f1"],
    },
    {
        "model": "TF-IDF + N-grams",
        "val_accuracy": tfidf_ngram_val_metrics["accuracy"],
        "val_macro_f1": tfidf_ngram_val_metrics["macro_f1"],
        "val_weighted_f1": tfidf_ngram_val_metrics["weighted_f1"],
        "test_accuracy": tfidf_ngram_test_metrics["accuracy"],
        "test_macro_f1": tfidf_ngram_test_metrics["macro_f1"],
        "test_weighted_f1": tfidf_ngram_test_metrics["weighted_f1"],
    },
    {
        "model": "Sentence-BERT + LR",
        "val_accuracy": sbert_val_metrics["accuracy"],
        "val_macro_f1": sbert_val_metrics["macro_f1"],
        "val_weighted_f1": sbert_val_metrics["weighted_f1"],
        "test_accuracy": sbert_test_metrics["accuracy"],
        "test_macro_f1": sbert_test_metrics["macro_f1"],
        "test_weighted_f1": sbert_test_metrics["weighted_f1"],
    },
    {
        "model": "BioBERT",
        "val_accuracy": biobert_val_metrics["accuracy"],
        "val_macro_f1": biobert_val_metrics["macro_f1"],
        "val_weighted_f1": biobert_val_metrics["weighted_f1"],
        "test_accuracy": biobert_test_metrics["accuracy"],
        "test_macro_f1": biobert_test_metrics["macro_f1"],
        "test_weighted_f1": biobert_test_metrics["weighted_f1"],
    }
])

results_summary = results_summary.sort_values(by=["test_macro_f1", "test_accuracy"], ascending=False).reset_index(drop=True)
results_summary


## Optional: answer-quality prediction task

This creates a binary target:
- `1` = model predicted correctly
- `0` = model predicted incorrectly

Below is an example using the **best model's prediction file** later.


In [ ]:
# Example using BioBERT outputs for Task 3
quality_df = biobert_test_out.copy()

quality_df["question_len"] = quality_df["question"].fillna("").apply(lambda x: len(str(x).split()))
quality_df["context_len"] = quality_df["context"].fillna("").apply(lambda x: len(str(x).split()))
quality_df["quality"] = quality_df["correct"].astype(int)

quality_df[["pmid", "question_len", "context_len", "true_label", "predicted_label", "quality"]].head()


## Files produced by this notebook

- `tfidf_test_predictions.csv`
- `tfidf_ngram_test_predictions.csv`
- `sbert_test_predictions.csv`
- `biobert_test_predictions.csv`
- `X_train_sbert.npy`, `X_val_sbert.npy`, `X_test_sbert.npy`

These are useful for your report, error analysis, and Task 3.
